# STFT Chunk Explorer — Raw vs Filtered

Run the **Setup** cell once, then repeatedly run the **Next chunk** cell.
Each run shows one chunk side-by-side: **raw** (left) vs **filtered** (right).

Filters applied:
- **C1** High-pass 400 Hz, 24 dB/Oct (4th-order Butterworth)
- **C4** Peaking bell 1009 Hz, +20 dB, Q=0.71


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor
from logic.filters import HighPassFilter, PeakingFilter

%matplotlib inline


In [ ]:
# --- Config ---
WAV_FILE      = '../data/static_10m_000.wav'
SAMPLE_RATE   = 44100
CHUNK_SIZE    = 8192
NPERSEG       = 512
NOVERLAP      = 256
OUTER_INDICES = slice(0, 4)
CHANNEL_NAMES = ['CH1 (outer, -x-y)', 'CH2 (outer, +x-y)',
                 'CH3 (outer, -x+y)', 'CH4 (outer, +x+y)']

# Filter params (from Parametric EQ screenshot)
HP_CUTOFF_HZ  = 400      # C1: high-pass, 24 dB/Oct
PEAK_FREQ_HZ  = 1009     # C4: peaking bell
PEAK_GAIN_DB  = 20.0     # C4: +20 dB
PEAK_Q        = 0.71     # C4: Q


In [ ]:
def make_stream(apply_filters: bool):
    loader = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
    audio  = loader.stream()
    if apply_filters:
        hp   = HighPassFilter(cutoff_hz=HP_CUTOFF_HZ, sampling_rate=SAMPLE_RATE, order=4)
        peak = PeakingFilter(center_hz=PEAK_FREQ_HZ, gain_db=PEAK_GAIN_DB, q=PEAK_Q, sampling_rate=SAMPLE_RATE)
        audio = peak.process(hp.process(audio))
    proc = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
    return proc.process(audio)

gen_raw      = make_stream(apply_filters=False)
gen_filtered = make_stream(apply_filters=True)
print('Generators ready — run the next cell to draw each chunk (raw vs filtered).')


In [ ]:
# ── Next chunk ── run this cell repeatedly ──────────────────────────────────
try:
    chunk_raw = next(gen_raw)
    chunk_flt = next(gen_filtered)
except StopIteration:
    print('No more chunks — file exhausted.')
    chunk_raw = chunk_flt = None

if chunk_raw is not None:
    plot_order = [2, 3, 0, 1]   # top: CH3, CH4 / bottom: CH1, CH2

    fig, axes = plt.subplots(4, 2, figsize=(14, 14), sharex=True, sharey=True)
    fig.suptitle(
        f'STFT — timestamp={chunk_raw.timestamp:.3f}s  '
        f'| frames={chunk_raw.magnitudes.shape[2]}  '
        f'| Left: raw   Right: HP {HP_CUTOFF_HZ}Hz + Peak {PEAK_FREQ_HZ}Hz +{PEAK_GAIN_DB:.0f}dB',
        fontsize=12
    )

    mag_raw = chunk_raw.magnitudes[OUTER_INDICES]
    mag_flt = chunk_flt.magnitudes[OUTER_INDICES]
    db_raw  = 20 * np.log10(mag_raw + 1e-6)
    db_flt  = 20 * np.log10(mag_flt + 1e-6)
    vmin = min(db_raw.min(), db_flt.min())
    vmax = max(db_raw.max(), db_flt.max())
    times_ms = chunk_raw.times * 1000

    for row, ch_idx in enumerate(plot_order):
        for col, (db, label) in enumerate([(db_raw, 'raw'), (db_flt, 'filtered')]):
            ax = axes[row, col]
            im = ax.pcolormesh(
                times_ms, chunk_raw.freqs, db[ch_idx],
                shading='auto', cmap='inferno', vmin=vmin, vmax=vmax
            )
            ax.set_title(f'{CHANNEL_NAMES[ch_idx]} — {label}', fontsize=9)
            ax.set_ylabel('Frequency (Hz)')
            ax.set_xlabel('Time (ms)')
            fig.colorbar(im, ax=ax, label='dB')

    plt.tight_layout()
    plt.show()
    print(f'timestamp={chunk_raw.timestamp:.3f}s  shape={mag_raw.shape}  (channels, freqs, frames)')
